# Batch Raster Clipping with a Vector Mask (GPKG)

This notebook clips every raster in an input folder to the extent/geometry of a vector layer
stored in a GeoPackage (`.gpkg`), and writes the clipped rasters to an output folder.

**Workflow**
1. Set the input raster folder, the GeoPackage path, and the output folder.
2. Load the vector mask with GeoPandas.
3. Loop over all rasters, reproject the mask to each raster's CRS if needed, clip with `rasterio.mask.mask`, and write the result.
4. Print a short summary / QA table at the end.

**Requirements:** `rasterio`, `geopandas`, `shapely` (install with `pip install rasterio geopandas` if missing).

In [ ]:
import os
import glob
from pathlib import Path

import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.warp import transform_geom


## 1. Configuration

Edit the paths below.

In [ ]:
# --- INPUT / OUTPUT PATHS -----------------------------------------------
RASTER_FOLDER = r"/Users/gregorygiuliani/Desktop/Basilicata/"      # folder containing the 11 rasters
GPKG_PATH     = r"/Users/gregorygiuliani/Desktop/output/Extent/basilicata_grid.gpkg"          # vector mask (GeoPackage)
OUTPUT_FOLDER = r"./clipped/"      # where clipped rasters will be written

# Optional: specific layer name inside the GeoPackage (None = first/only layer)
GPKG_LAYER = None

# Raster file extensions to look for in RASTER_FOLDER
RASTER_EXTENSIONS = (".tif", ".tiff")

# If True, clip strictly to the geometry outline (pixels outside the polygon
# become nodata). If False, clip to the bounding box of the geometry only.
CROP_TO_GEOMETRY = True

# nodata value to use for pixels outside the mask, if the raster has none defined.
# Set to None to keep the raster's own nodata (recommended when it is already set).
FALLBACK_NODATA = 0
# -------------------------------------------------------------------------

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
print(f"Output folder ready: {OUTPUT_FOLDER}")


## 2. Load the vector mask

In [ ]:
vector_mask = gpd.read_file(GPKG_PATH, layer=GPKG_LAYER)

if vector_mask.empty:
    raise ValueError("The vector mask has no features. Check the GeoPackage / layer name.")

print(f"Loaded {len(vector_mask)} feature(s) from: {GPKG_PATH}")
print(f"Vector CRS: {vector_mask.crs}")
vector_mask.head()


## 3. Discover input rasters

In [ ]:
raster_paths = sorted(
    p for p in glob.glob(os.path.join(RASTER_FOLDER, "*"))
    if p.lower().endswith(RASTER_EXTENSIONS)
)

print(f"Found {len(raster_paths)} raster(s):")
for p in raster_paths:
    print(" -", os.path.basename(p))


## 4. Clipping function

In [ ]:
def clip_raster_with_vector(raster_path, mask_gdf, output_path,
                             crop=True, fallback_nodata=None):
    """Clip a single raster to the geometries in mask_gdf and write the result.

    The vector mask is reprojected to the raster's CRS if the CRSs differ.
    """
    with rasterio.open(raster_path) as src:
        # Reproject mask geometries to the raster CRS if needed
        if mask_gdf.crs != src.crs:
            geoms = [
                transform_geom(mask_gdf.crs, src.crs, geom.__geo_interface__)
                for geom in mask_gdf.geometry
            ]
        else:
            geoms = [geom.__geo_interface__ for geom in mask_gdf.geometry]

        nodata_value = src.nodata if src.nodata is not None else fallback_nodata

        out_image, out_transform = mask(
            src,
            geoms,
            crop=crop,
            nodata=nodata_value,
            filled=True,
        )

        out_meta = src.meta.copy()
        out_meta.update({
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform,
            "nodata": nodata_value,
        })

    with rasterio.open(output_path, "w", **out_meta) as dst:
        dst.write(out_image)

    return out_image.shape


## 5. Run the clipping loop

In [ ]:
results = []

for raster_path in raster_paths:
    fname = os.path.basename(raster_path)
    out_path = os.path.join(OUTPUT_FOLDER, Path(fname).stem + "_clipped.tif")

    try:
        shape = clip_raster_with_vector(
            raster_path,
            vector_mask,
            out_path,
            crop=CROP_TO_GEOMETRY,
            fallback_nodata=FALLBACK_NODATA,
        )
        results.append((fname, "OK", shape, out_path))
        print(f"[OK]   {fname} -> {os.path.basename(out_path)}  shape={shape}")
    except Exception as e:
        results.append((fname, f"FAILED: {e}", None, None))
        print(f"[FAIL] {fname}: {e}")


## 6. Summary

In [ ]:
import pandas as pd

summary_df = pd.DataFrame(results, columns=["raster", "status", "output_shape", "output_path"])
n_ok = (summary_df["status"] == "OK").sum()
print(f"Clipped {n_ok}/{len(summary_df)} rasters successfully.")
summary_df
